# Individual Challenge - Data Cleaning Expert: Lloyd Anthony D'Silva: 64858

### 1. Installing and Importing necessary Libraries

In [ ]:
!pip install fuzzywuzzy
!pip install python-Levenshtein
!pip install pandas

In [1]:
import pandas as pd
import numpy as np
import re
from fuzzywuzzy import fuzz
from datetime import datetime, timedelta

### 2. Accesing the Dataset - Loading Original Dataset

In [2]:
ORIGINAL_DATASET_PATH = 'GlobalWeatherRepository.csv'
MESSY_DATASET_PATH = 'GlobalWeatherRepository_messy.csv'

# Load Original Dataset and Create Messy Copy
try:
    df_original = pd.read_csv(ORIGINAL_DATASET_PATH)
    df_messy = df_original.copy()
    print(f"Original dataset loaded successfully from '{ORIGINAL_DATASET_PATH}'. Shape: {df_original.shape}")
except FileNotFoundError:
    print(f"Error: Original dataset '{ORIGINAL_DATASET_PATH}' not found. Please ensure it's in the same directory.")
    exit()

Original dataset loaded successfully from 'GlobalWeatherRepository.csv'. Shape: (40541, 10)


In [3]:
df_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40541 entries, 0 to 40540
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Country              40541 non-null  object 
 1   location_name        40541 non-null  object 
 2   latitude             40541 non-null  float64
 3   longitude            40541 non-null  float64
 4   last_updated         40541 non-null  object 
 5   temperature_celsius  40541 non-null  float64
 6   precip_mm            40541 non-null  float64
 7   humidity             40541 non-null  int64  
 8   air_quality_PM2.5    40541 non-null  float64
 9   air_quality_PM10     40541 non-null  float64
dtypes: float64(6), int64(1), object(3)
memory usage: 3.1+ MB


### 3. Creation of Messy Dataset and Introducing Data Quality Issues into the Dataset

#### 3.1. Introducing Missing values in Numerical and Categorical Columns

In [4]:
# Introduce Data Quality Issues into the Messy Dataset
print("\nIntroducing various data quality issues")

# Introducing Missing Values 
print("1. Introducing missing values")
# Randomly replace values with NaN
for col in ['temperature_celsius', 'precip_mm', 'humidity', 'air_quality_PM2.5', 'air_quality_PM10']:
    df_messy.loc[df_messy.sample(frac=0.05).index, col] = np.nan # 5% NaN

# Introduce missing values in categorical columns
missing_placeholders = ['N/A', '-', 'UNKNOWN', 'NULL']
for col in ['Country', 'location_name']:
    # Replace some values with placeholders
    for _ in range(int(len(df_messy) * 0.02)): # 2% missing
        idx = np.random.randint(0, len(df_messy))
        df_messy.loc[idx, col] = np.random.choice(missing_placeholders)


Introducing various data quality issues
1. Introducing missing values


#### 3.2. Introducing Outliers, Inconsistenet Data Formats and Types

In [5]:
# Introducing Outliers and Anomalous Values
print("2. Introducing outliers and anomalous values")

# temperature_celsius: Physically impossible extremes
temp_outlier_indices = df_messy.sample(frac=0.01).index
for idx in temp_outlier_indices:
    df_messy.loc[idx, 'temperature_celsius'] = np.random.choice([1000, -200])

# precip_mm: Unusually high precipitation
precip_outlier_indices = df_messy.sample(frac=0.01).index
for idx in precip_outlier_indices:
    df_messy.loc[idx, 'precip_mm'] = np.random.uniform(500, 5000)

# air_quality_PM2.5, air_quality_PM10: Negative values, PM2.5 > PM10
aq_outlier_indices = df_messy.sample(frac=0.02).index
for idx in aq_outlier_indices:
    if np.random.rand() < 0.5: # Negative values
        df_messy.loc[idx, 'air_quality_PM2.5'] = -np.random.uniform(1, 10)
        df_messy.loc[idx, 'air_quality_PM10'] = -np.random.uniform(1, 10)
    else: # PM2.5 > PM10
        if pd.notna(df_messy.loc[idx, 'air_quality_PM2.5']) and pd.notna(df_messy.loc[idx, 'air_quality_PM10']):
            df_messy.loc[idx, 'air_quality_PM2.5'] = df_messy.loc[idx, 'air_quality_PM10'] + np.random.uniform(1, 50)
        else: # If one is NaN, make them inconsistent
            df_messy.loc[idx, 'air_quality_PM2.5'] = 100
            df_messy.loc[idx, 'air_quality_PM10'] = 50

2. Introducing outliers and anomalous values


In [6]:
# Introducing Inconsistent Data Formats and Types 
print("3. Introducing inconsistent formats and types")

# temperature_celsius: Convert some to Fahrenheit, add non-numeric strings
temp_indices = df_messy.sample(frac=0.03).index
for idx in temp_indices:
    if pd.notna(df_messy.loc[idx, 'temperature_celsius']):
        if np.random.rand() < 0.5: # 50% chance to convert to Fahrenheit
            df_messy.loc[idx, 'temperature_celsius'] = f"{df_messy.loc[idx, 'temperature_celsius'] * 9/5 + 32:.1f}F"
        else: # 50% chance to add non-numeric string
            df_messy.loc[idx, 'temperature_celsius'] = np.random.choice(['Freezing', 'Hot', 'Mild'])

3. Introducing inconsistent formats and types


C:\Users\Admin\AppData\Local\Temp\ipykernel_34484\2586842688.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '46.2F' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_messy.loc[idx, 'temperature_celsius'] = f"{df_messy.loc[idx, 'temperature_celsius'] * 9/5 + 32:.1f}F"


#### 3.3 Introducing Non-numeric entries, illogical negative values into the precip_mm column

In [7]:
# precip_mm: Add non-numeric entries, illogical negative values
print("4. Introducing non-numeric entries, illogical negative values in precip_mm")

precip_indices = df_messy.sample(frac=0.03).index
for idx in precip_indices:
    if np.random.rand() < 0.5:
        df_messy.loc[idx, 'precip_mm'] = np.random.choice(['trace', 'heavy rain', 'light drizzle'])
    else:
        df_messy.loc[idx, 'precip_mm'] = -abs(df_messy.loc[idx, 'precip_mm']) if pd.notna(df_messy.loc[idx, 'precip_mm']) else -1.0


4. Introducing non-numeric entries, illogical negative values in precip_mm


C:\Users\Admin\AppData\Local\Temp\ipykernel_34484\1421986377.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'light drizzle' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_messy.loc[idx, 'precip_mm'] = np.random.choice(['trace', 'heavy rain', 'light drizzle'])


#### 3.4. Introducing Mixed data formats, invalid date components and future/past dates for last_updated column

In [8]:
# For last_updated column introduce: Mixed date formats, invalid date components, future/past dates
date_formats = date_formats = [
    "%Y/%m/%d %H:%M",  # Matching the existing format
    "%Y-%m-%d %H:%M",  # Hyphen-separated version
    "%d/%m/%Y %H:%M",  # European format
    "%m/%d/%Y %H:%M",  # US format
    "%Y/%m/%d",        # Date only, no time
    "%Y-%m-%d",        # Hyphen-separated date only
]

date_indices = df_messy.sample(frac=0.05).index
for idx in date_indices:
    original_date = pd.to_datetime(df_original.loc[idx, 'last_updated'], errors='coerce')
    if pd.notna(original_date):
        if np.random.rand() < 0.4: # Mixed formats
            df_messy.loc[idx, 'last_updated'] = original_date.strftime(np.random.choice(date_formats))
        elif np.random.rand() < 0.7: # Invalid components (e.g., month 13, day 40)
            df_messy.loc[idx, 'last_updated'] = f"{original_date.year}/13/40 00:00"
        else: # Future/Past dates
            if np.random.rand() < 0.5:
                df_messy.loc[idx, 'last_updated'] = (datetime(2050, 1, 1) + timedelta(days=np.random.randint(1, 365))).strftime("%Y-%m-%d %H:%M:%S")
            else:
                df_messy.loc[idx, 'last_updated'] = (datetime(1900, 1, 1) + timedelta(days=np.random.randint(1, 365))).strftime("%Y-%m-%d %H:%M:%S")

#### 3.5. Introducing converted value, units and commas for air_qaulity_PM2.5 and air_quality_PM10 columns

In [9]:
# For air_quality_PM2.5, air_quality_PM10 introduce converted value, units, commas
aq_indices = df_messy.sample(frac=0.05).index
for idx in aq_indices:
    for col in ['air_quality_PM2.5', 'air_quality_PM10']:
        if pd.notna(df_messy.loc[idx, col]):
            val = df_messy.loc[idx, col]
            if np.random.rand() < 0.3:
                df_messy.loc[idx, col] = f"({val:.1f})-converted value"
            elif np.random.rand() < 0.6:
                df_messy.loc[idx, col] = f"{val:.1f} ug/m3"
            else:
                df_messy.loc[idx, col] = f"{val:,.1f}" # Add commas

# latitude, longitude: Values outside valid ranges, non-numeric characters, swapped
geo_indices = df_messy.sample(frac=0.03).index
for idx in geo_indices:
    if np.random.rand() < 0.3: # Out of range
        df_messy.loc[idx, 'latitude'] = np.random.uniform(91, 180)
        df_messy.loc[idx, 'longitude'] = np.random.uniform(-181, -360)
    elif np.random.rand() < 0.6: # Non-numeric
        df_messy.loc[idx, 'latitude'] = 'invalid_lat'
        df_messy.loc[idx, 'longitude'] = 'invalid_lon'
    else: # Swapped
        df_messy.loc[idx, 'latitude'], df_messy.loc[idx, 'longitude'] = \
            df_messy.loc[idx, 'longitude'], df_messy.loc[idx, 'latitude']

C:\Users\Admin\AppData\Local\Temp\ipykernel_34484\2259645534.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.1' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_messy.loc[idx, col] = f"{val:,.1f}" # Add commas
C:\Users\Admin\AppData\Local\Temp\ipykernel_34484\2259645534.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2.7 ug/m3' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_messy.loc[idx, col] = f"{val:.1f} ug/m3"
C:\Users\Admin\AppData\Local\Temp\ipykernel_34484\2259645534.py:21: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'invalid_lat' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_messy.lo

#### 3.6. Introducing Duplicate records and near duplicate records

In [10]:
# Introduce Duplicate Records
print("Introducing duplicate records (exact and near-duplicates)")

# Exact duplicates
df_messy = pd.concat([df_messy, df_messy.sample(n=50, random_state=42)], ignore_index=True)

# Near-duplicates (minor variations)
for _ in range(30): # Introduce 30 near-duplicates
    idx = np.random.randint(0, len(df_messy))
    row = df_messy.iloc[idx].copy()
    if np.random.rand() < 0.5: # Vary location_name
        row['location_name'] = row['location_name'].lower() if isinstance(row['location_name'], str) else row['location_name']
        if isinstance(row['location_name'], str) and len(row['location_name']) > 3:
            row['location_name'] = row['location_name'][:-1] + np.random.choice(['a', 'z', 'x'])
    else: # Vary Country
        row['Country'] = row['Country'].upper() if isinstance(row['Country'], str) else row['Country']
        if isinstance(row['Country'], str) and len(row['Country']) > 3:
            row['Country'] = row['Country'] + ' (typo)'
    df_messy = pd.concat([df_messy, row.to_frame().T], ignore_index=True)

Introducing duplicate records (exact and near-duplicates)


3.7. Introducing misspellings, inconsistent, capitalization/spacing for Country and Location 

In [11]:
# Introduce misspellings, inconsistent capitalization/spacing
typo_map_country = {'Albania': 'Albana', 'United States': 'United States of America', 'Australia': 'Austraila'}
typo_map_location = {'Tirana': 'Tiranaa', 'New York': 'NYC', 'London': 'london'}

for idx in df_messy.sample(frac=0.05).index:
    if df_messy.loc[idx, 'Country'] in typo_map_country:
        df_messy.loc[idx, 'Country'] = typo_map_country[df_messy.loc[idx, 'Country']]
    if df_messy.loc[idx, 'location_name'] in typo_map_location:
        df_messy.loc[idx, 'location_name'] = typo_map_location[df_messy.loc[idx, 'location_name']]
    
    # Inconsistent casing and extra spaces
    if isinstance(df_messy.loc[idx, 'Country'], str):
        df_messy.loc[idx, 'Country'] = df_messy.loc[idx, 'Country'].upper() + '  '
    if isinstance(df_messy.loc[idx, 'location_name'], str):
        df_messy.loc[idx, 'location_name'] = '  ' + df_messy.loc[idx, 'location_name'].lower()

# Irrelevant entries
irrelevant_entries = pd.DataFrame()
df_messy = pd.concat([df_messy, irrelevant_entries], ignore_index=True)

#### 3.8. Saving the messy dataset to csv

In [12]:
# Save the messy dataset
df_messy.to_csv(MESSY_DATASET_PATH, index=False)
print(f"\nMessy dataset saved to '{MESSY_DATASET_PATH}'. New shape: {df_messy.shape}")
print("Initial messy data head:")
df_messy


Messy dataset saved to 'GlobalWeatherRepository_messy.csv'. New shape: (40621, 10)
Initial messy data head:


,Country,location_name,latitude,longitude,last_updated,temperature_celsius,precip_mm,humidity,air_quality_PM2.5,air_quality_PM10
0,Afghanistan,Kabul,34.52,69.18,2023/8/29 14:00,28.8,0.0,19.0,NaN,11.1
1,Albania,Tirana,41.33,19.82,2023/8/29 11:30,Hot,0.0,54.0,28.2,29.6
2,Algeria,Algiers,36.76,3.05,2023/8/29 10:30,28.0,0.0,30.0,NaN,7.9
3,Andorra,Andorra La Vella,42.5,1.52,2023/8/29 11:30,NaN,0.0,51.0,0.5,0.8
4,Angola,Luanda,-8.84,13.23,2023/8/29 10:30,25.0,0.0,69.0,139.6,203.3
...,...,...,...,...,...,...,...,...,...,...
40616,Vanuatu,port vilz,-17.73,168.32,2023/9/23 9:15,NaN,0.0,74.0,2.2,5.4
40617,MADAGASCAR (typo),Ivory,-24.37,46.45,2024/1/18 21:00,24.1,2.66,95.0,1.3,2.3
40618,Fiji Islands,suva,-18.13,178.42,2023/11/5 8:15,28.0,0.0,84.0,4.7 ug/m3,7.7 ug/m3
40619,MOROCCO,rabax,34.03,-6.84,2024/3/10 16:45,17.0,0.0,55.0,NaN,4.4


### 4. Accessing the messy dataset

In [13]:
print("\nInitial messy data info:")
df_messy.info()


Initial messy data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40621 entries, 0 to 40620
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Country              40621 non-null  object
 1   location_name        40621 non-null  object
 2   latitude             40621 non-null  object
 3   longitude            40621 non-null  object
 4   last_updated         40621 non-null  object
 5   temperature_celsius  38605 non-null  object
 6   precip_mm            38664 non-null  object
 7   humidity             38592 non-null  object
 8   air_quality_PM2.5    38628 non-null  object
 9   air_quality_PM10     38638 non-null  object
dtypes: object(10)
memory usage: 3.1+ MB


In [14]:
df_cleaned = pd.read_csv(MESSY_DATASET_PATH)

### 5. Data Cleaning Process

#### 5.1. Standardization: Textual & Categorical Normalization

In [15]:
print("\nConverting disguised missing values to NaN...")
# DO NOT include empty string in disguised values
disguised_missing_values = ['N/A', '-', 'UNKNOWN', 'NULL']
for col in ['Country', 'location_name']:
    print(f"\nUnique values in '{col}' before replacement:")
    print(df_cleaned[col].value_counts(dropna=False).head(10))  # Debug
    
    df_cleaned[col] = df_cleaned[col].replace(disguised_missing_values, np.nan)
    
    print(f"\nConverted disguised missing values in '{col}' to NaN.")
    print(f"Remaining NaNs in '{col}': {df_cleaned[col].isna().sum()}")



Converting disguised missing values to NaN...

Unique values in 'Country' before replacement:
Country
Bulgaria      513
Sudan         397
NaN           396
Indonesia     393
Madagascar    391
Thailand      386
Bolivia       385
Turkey        385
Belgium       356
Russia        310
Name: count, dtype: int64

Converted disguised missing values in 'Country' to NaN.
Remaining NaNs in 'Country': 767

Unique values in 'location_name' before replacement:
location_name
NaN          366
-            206
Baku         202
Belmopan     201
Abu Dhabi    201
Amman        201
Cairo        200
Monrovia     200
Freetown     200
Sanaa        200
Name: count, dtype: int64

Converted disguised missing values in 'location_name' to NaN.
Remaining NaNs in 'location_name': 752


In [16]:
print("Textual & Categorical Normalization (Country, location_name)")

# Remove extra spaces and standardize
for col in ['Country', 'location_name']:
    df_cleaned[col] = df_cleaned[col].apply(lambda x: x.strip().title() if isinstance(x, str) else x)
    # df_cleaned[col].astype(str).str.strip().str.title()

# Manual mapping for common misspellings/abbreviations
country_corrections = {
    'Albana': 'Albania',
    'United States Of America': 'United States',
    'Austraila': 'Australia',
    'Usa': 'United States',
    'Uk': 'United Kingdom'
}
location_corrections = {
    'Tiranaa': 'Tirana',
    'Nyc': 'New York City',
    'London': 'London' 
}

df_cleaned['Country'] = df_cleaned['Country'].replace(country_corrections)
df_cleaned['location_name'] = df_cleaned['location_name'].replace(location_corrections)

df_cleaned['Country'].describe()

Textual & Categorical Normalization (Country, location_name)


count        39854
unique         202
top       Bulgaria
freq           548
Name: Country, dtype: object

5.2. Data Type Conversion and Format Standardization

In [17]:
print("Data Type Conversion and Format Standardization")

# Function to clean and convert numerical columns (handling various formats)
def clean_and_convert_numeric(series):
    cleaned_series = series.astype(str).str.replace(',', '').str.strip()
    # Extract numbers from '(value)-converted value' and 'value unit' formats
    cleaned_series = cleaned_series.apply(lambda x: re.search(r'\(?(\-?\d+\.?\d*)\)?', x).group(1) if re.search(r'\(?(\-?\d+\.?\d*)\)?', x) else x)
    # Convert Fahrenheit to Celsius (simple rule: if value > 50 and ends with F)
    cleaned_series = cleaned_series.apply(lambda x: (float(x[:-1]) - 32) * 5/9 if isinstance(x, str) and x.endswith('F') else x)
    return pd.to_numeric(cleaned_series, errors='coerce')

df_cleaned['temperature_celsius'] = clean_and_convert_numeric(df_cleaned['temperature_celsius'])
df_cleaned['precip_mm'] = clean_and_convert_numeric(df_cleaned['precip_mm'])
df_cleaned['humidity'] = clean_and_convert_numeric(df_cleaned['humidity'])
df_cleaned['air_quality_PM2.5'] = clean_and_convert_numeric(df_cleaned['air_quality_PM2.5'])
df_cleaned['air_quality_PM10'] = clean_and_convert_numeric(df_cleaned['air_quality_PM10'])

# Convert 'last_updated' to datetime
df_cleaned['last_updated'] = pd.to_datetime(df_cleaned['last_updated'], errors='coerce')

Data Type Conversion and Format Standardization


In [18]:
# Validate and correct latitude/longitude
def validate_lat_lon(df):
    # Convert to numeric, coercing errors
    df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
    df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

    # Identify and attempt to correct swapped lat/lon (simple check: if lat is out of range but lon is in lat range)
    invalid_lat_mask = (df['latitude'].isna()) | (df['latitude'] < -90) | (df['latitude'] > 90)
    invalid_lon_mask = (df['longitude'].isna()) | (df['longitude'] < -180) | (df['longitude'] > 180)

    swapped_mask = invalid_lat_mask & (~invalid_lon_mask) & \
                   (df['longitude'] >= -90) & (df['longitude'] <= 90) & \
                   (df['latitude'].apply(lambda x: -180 <= x <= 180 if pd.notna(x) else False))

    df.loc[swapped_mask, ['latitude', 'longitude']] = df.loc[swapped_mask, ['longitude', 'latitude']].values

    # After potential swap, re-check and set remaining invalid to NaN
    df.loc[(df['latitude'] < -90) | (df['latitude'] > 90), 'latitude'] = np.nan
    df.loc[(df['longitude'] < -180) | (df['longitude'] > 180), 'longitude'] = np.nan
    
    return df

df_cleaned = validate_lat_lon(df_cleaned)

In [19]:
# Handling Missing Data: Imputation and Deletion
print("\nHandling Missing Data")

# Identify and count explicit and disguised missing values
initial_missing_count = df_cleaned.isnull().sum().sum()
print(f"Initial NaN count: {initial_missing_count}")

# Impute numerical columns with median
for col in ['temperature_celsius', 'precip_mm', 'humidity', 'air_quality_PM2.5', 'air_quality_PM10', 'latitude', 'longitude']:
    if df_cleaned[col].isnull().any():
        median_val = df_cleaned[col].median()
        df_cleaned[col] = df_cleaned[col].fillna(median_val)
        print(f"  - Imputed missing values in '{col}' with median: {median_val}")

for col in ['Country', 'location_name']:
    if col in df_cleaned.columns:
        # Drop rows where Country is missing or "Unknown"
        original_len = len(df_cleaned)
        df_cleaned[col] = df_cleaned[col].str.replace(r'\s*\(Typo\)', '', regex=True)

        df_cleaned = df_cleaned[~df_cleaned[col].isin([np.nan, 'Unknown', 'N/A', '-', 'UNKNOWN', 'NULL', 'NaN', 'Null']) & df_cleaned[col].notnull()]
        new_len = len(df_cleaned)
        print(f"  - Dropped {original_len - new_len} rows where '{col}' was missing or 'Unknown'.")

# For 'last_updated', if still NaN after conversion, drop or impute with a sensible default (e.g., most common date or a fixed date)
# For this case, we'll drop rows where 'last_updated' is still NaN after conversion, as it's a critical timestamp.
df_cleaned.dropna(subset=['last_updated'], inplace=True)
print(f"  - Dropped rows with invalid 'last_updated' timestamps. New shape: {df_cleaned.shape}")


Handling Missing Data
Initial NaN count: 16367
  - Imputed missing values in 'temperature_celsius' with median: 22.0
  - Imputed missing values in 'precip_mm' with median: 0.0
  - Imputed missing values in 'humidity' with median: 75.0
  - Imputed missing values in 'air_quality_PM2.5' with median: 7.4
  - Imputed missing values in 'air_quality_PM10' with median: 12.7
  - Imputed missing values in 'latitude' with median: 17.25
  - Imputed missing values in 'longitude' with median: 23.24
  - Dropped 798 rows where 'Country' was missing or 'Unknown'.
  - Dropped 787 rows where 'location_name' was missing or 'Unknown'.
  - Dropped rows with invalid 'last_updated' timestamps. New shape: (37222, 10)


In [20]:
# Outlier Detection and Treatment
print("Outlier Detection and Treatment")

# Function to detect and treat outliers using IQR
def iqr_outlier_treatment(df, column, cap_method='median'):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_count = df[(df[column] < lower_bound) | (df[column] > upper_bound)].shape[0]
    if outlier_count > 0:
        print(f"  - Detected {outlier_count} outliers in '{column}' (IQR method).")
        if cap_method == 'median':
            median_val = df[column].median()
            df.loc[df[column] < lower_bound, column] = median_val
            df.loc[df[column] > upper_bound, column] = median_val
            print(f"    - Capped outliers in '{column}' using median.")
        elif cap_method == 'bounds':
            df.loc[df[column] < lower_bound, column] = lower_bound
            df.loc[df[column] > upper_bound, column] = upper_bound
            print(f"    - Capped outliers in '{column}' to IQR bounds.")
    return df

Outlier Detection and Treatment


In [21]:
for col in ['temperature_celsius', 'precip_mm', 'humidity', 'air_quality_PM2.5', 'air_quality_PM10']:
    df_cleaned = iqr_outlier_treatment(df_cleaned, col, cap_method='bounds')

# Domain-specific rules for physical impossibilities
# Temperature: -100 to 60 Celsius
df_cleaned.loc[(df_cleaned['temperature_celsius'] < -100) | (df_cleaned['temperature_celsius'] > 60), 'temperature_celsius'] = np.nan
# Precip_mm: Must be non-negative
df_cleaned.loc[df_cleaned['precip_mm'] < 0, 'precip_mm'] = 0
# Humidity: 0 to 100
df_cleaned.loc[(df_cleaned['humidity'] < 0) | (df_cleaned['humidity'] > 100), 'humidity'] = np.nan
# Air Quality: Must be non-negative
df_cleaned.loc[df_cleaned['air_quality_PM2.5'] < 0, 'air_quality_PM2.5'] = 0
df_cleaned.loc[df_cleaned['air_quality_PM10'] < 0, 'air_quality_PM10'] = 0

  - Detected 1208 outliers in 'temperature_celsius' (IQR method).
    - Capped outliers in 'temperature_celsius' to IQR bounds.
  - Detected 8557 outliers in 'precip_mm' (IQR method).
    - Capped outliers in 'precip_mm' to IQR bounds.
  - Detected 1132 outliers in 'humidity' (IQR method).
    - Capped outliers in 'humidity' to IQR bounds.
  - Detected 4246 outliers in 'air_quality_PM2.5' (IQR method).
    - Capped outliers in 'air_quality_PM2.5' to IQR bounds.
  - Detected 5053 outliers in 'air_quality_PM10' (IQR method).
    - Capped outliers in 'air_quality_PM10' to IQR bounds.


In [22]:
# Relational outlier: PM2.5 > PM10
pm_relational_outliers = df_cleaned[df_cleaned['air_quality_PM2.5'] > df_cleaned['air_quality_PM10']].shape[0]
if pm_relational_outliers > 0:
    df_cleaned.loc[df_cleaned['air_quality_PM2.5'] > df_cleaned['air_quality_PM10'], 'air_quality_PM2.5'] = \
        df_cleaned.loc[df_cleaned['air_quality_PM2.5'] > df_cleaned['air_quality_PM10'], 'air_quality_PM10']
    print(f"  - Corrected {pm_relational_outliers} relational outliers where PM2.5 > PM10.")

# Re-impute any NaNs created by domain-specific rules
for col in ['temperature_celsius', 'humidity', 'air_quality_PM2.5', 'air_quality_PM10']:
    if df_cleaned[col].isnull().any():
        median_val = df_cleaned[col].median()
        df_cleaned[col] = df_cleaned[col].fillna(median_val)
        print(f"Re-imputed new NaNs in '{col}' with median: {median_val}")

  - Corrected 1570 relational outliers where PM2.5 > PM10.


In [23]:
duplicate_count = df_cleaned.duplicated().sum()
print(f"Total duplicate rows: {duplicate_count}")

Total duplicate rows: 65


In [24]:
# Duplicate Resolution
print("Duplicate Resolution")

# Remove exact duplicates
initial_rows = df_cleaned.shape[0]
df_cleaned.drop_duplicates(inplace=True)
exact_duplicates_removed = initial_rows - df_cleaned.shape[0]
print(f"  - Removed {exact_duplicates_removed} exact duplicate rows.")

# Handle near-duplicates using fuzzy matching for Country and location_name
def fuzzy_deduplicate(df, key_cols, threshold=90):
    unique_keys = {}
    canonical_indices = []
    
    # Create a temporary DataFrame to iterate over, ensuring string types for fuzzy matching
    temp_df = df[key_cols].astype(str)

    for idx, row in temp_df.iterrows():
        key_values = tuple(row[col] for col in key_cols)
        found_match = False
        for canonical_key, original_idx_list in unique_keys.items():
            # Check similarity for each key column
            all_match = True
            for i, col_val in enumerate(key_values):
                if fuzz.ratio(col_val.lower(), canonical_key[i].lower()) < threshold:
                    all_match = False
                    break
            if all_match:
                unique_keys[canonical_key].append(idx)
                found_match = True
                break
        if not found_match:
            unique_keys[key_values] = [idx]

    for canonical_key, original_idx_list in unique_keys.items():
        canonical_indices = [original_idx_list[0] for original_idx_list in unique_keys.values()]
        # canonical_indices.append(original_idx_list) # Keep the first one encountered
    
    return df.loc[canonical_indices].reset_index(drop=True)

# Apply fuzzy deduplication on 'Country' and 'location_name'
initial_rows_after_exact_dedup = df_cleaned.shape[0]
df_cleaned = fuzzy_deduplicate(df_cleaned, ['Country', 'location_name'], threshold=85)
near_duplicates_removed = initial_rows_after_exact_dedup - df_cleaned.shape[0]
print(f"  - Removed approximately {near_duplicates_removed} near-duplicate rows based on Country/Location fuzzy matching.")


Duplicate Resolution
  - Removed 65 exact duplicate rows.
  - Removed approximately 36929 near-duplicate rows based on Country/Location fuzzy matching.


In [25]:
# Temporal Data Validation and Correction
print("\n Temporal Data Validation and Correction")

# Filter out dates outside a reasonable range (e.g., 1998-01-01 to 2024-12-31)
min_date = datetime(1998, 1, 1)
max_date = datetime(2024, 12, 31)
invalid_date_count = df_cleaned[(df_cleaned['last_updated'] < min_date) | (df_cleaned['last_updated'] > max_date)].shape[0]
if invalid_date_count > 0:
    df_cleaned = df_cleaned[(df_cleaned['last_updated'] >= min_date) & (df_cleaned['last_updated'] <= max_date)]
    print(f"  - Removed {invalid_date_count} records with dates outside the valid range (1998-2024). New shape: {df_cleaned.shape}")

df_cleaned['last_updated_date'] = df_cleaned['last_updated'].dt.date  

# For multiple entries for the same Country/location_name on the same day, keep the most recent
df_cleaned.sort_values(by=['Country', 'location_name', 'last_updated'], inplace=True)
df_cleaned.drop_duplicates(subset=['Country', 'location_name', 'last_updated_date'], keep='last', inplace=True)
print("  - Kept only the most recent record for each Country/Location per day.")

# 3.7. Remove Irrelevant Entries
print("\n Removing Irrelevant Entries")
# Define patterns for irrelevant entries
irrelevant_country_patterns = []
irrelevant_location_patterns = []

initial_rows_before_irrelevant_removal = df_cleaned.shape[0]
df_cleaned = df_cleaned[~df_cleaned['Country'].isin(irrelevant_country_patterns)]
df_cleaned = df_cleaned[~df_cleaned['location_name'].isin(irrelevant_location_patterns)]
irrelevant_removed_count = initial_rows_before_irrelevant_removal - df_cleaned.shape[0]
print(f"  - Removed {irrelevant_removed_count} rows identified as irrelevant entries.")


 Temporal Data Validation and Correction
  - Kept only the most recent record for each Country/Location per day.

 Removing Irrelevant Entries
  - Removed 0 rows identified as irrelevant entries.


### 6. Post Data Cleaning Results

In [26]:
# Post-Cleaning Data Quality Assessment
print("\nPost-Cleaning Data Quality Assessment")

# Display cleaned data as a table
print("\nCleaned Data Head:")
print(df_cleaned.head().to_string())

# Display cleaned data info
print("\nCleaned Data Info:")
df_cleaned.info()

# Display missing values after cleaning
print("\nMissing Values After Cleaning:")
print(df_cleaned.isnull().sum().to_frame(name='Missing Count'))


Post-Cleaning Data Quality Assessment

Cleaned Data Head:
       Country     location_name  latitude  longitude        last_updated  temperature_celsius  precip_mm  humidity  air_quality_PM2.5  air_quality_PM10 last_updated_date
0  Afghanistan             Kabul     34.52      69.18 2023-08-29 14:00:00                 28.8        0.0      21.0               7.40              11.1        2023-08-29
1      Albania            Tirana     41.33      19.82 2023-08-29 11:30:00                 22.0        0.0      54.0              28.20              29.6        2023-08-29
2      Algeria           Algiers     36.76       3.05 2023-08-29 10:30:00                 28.0        0.0      30.0               7.40               7.9        2023-08-29
3      Andorra  Andorra La Vella     42.50       1.52 2023-08-29 11:30:00                 22.0        0.0      51.0               0.50               0.8        2023-08-29
4       Angola            Luanda     -8.84      13.23 2023-08-29 10:30:00             

In [27]:
# Verify PM2.5 <= PM10 constraint
pm_constraint_violations = df_cleaned[df_cleaned['air_quality_PM2.5'] > df_cleaned['air_quality_PM10']].shape
print(f"\n PM2.5 > PM10 violations after cleaning: {pm_constraint_violations}")

# Save the cleaned dataset
CLEANED_DATASET_PATH = 'GlobalWeatherRepository_cleaned.csv'
df_cleaned.to_csv(CLEANED_DATASET_PATH, index=False)
print(f"\n Cleaned dataset saved to '{CLEANED_DATASET_PATH}'. Final shape: {df_cleaned.shape}")

print("\n Data cleaning process completed.")


 PM2.5 > PM10 violations after cleaning: (0, 11)

 Cleaned dataset saved to 'GlobalWeatherRepository_cleaned.csv'. Final shape: (228, 11)

 Data cleaning process completed.
